# Canadian Wildfires in 2023

## Visualization of the data in an interactive Map

This notebooks produces an interactive Folium visualization of the recorded uncontrolled wildfire events in Canada in 2023 and their impact on the most relevant cities in Canada.

Therefore it combines three togglable layers 
* **Heatmap** with all uncontrolled NFDB fires in 2023, weighted by size class
* **Cities as CircleMarkers** coloured by their fire-impact score
* **Live heatmap** of recent active fires retrieved from the NASA FIRMS API

**Short summary of the following steps**
> 0. Import & Constants
> 1. Computation of GeoDataFrames & CRS Reprojection
> 2. Spatial Join & Fire Scores fore Cities
> 3. Classification of cities impact levels
> 4. Live fire data from NASA FIRMS
> 5. Put together the map layers
> 6. Export of the final map

**IMPORTANT!!** an active internet connection is necessary to run this notebooks, since 
otherwise the NASA FIRMS data can't be loaded. 

## 0. Import & Constants

In [1]:
import requests
from datetime import date
from pathlib import Path

import geopandas as gpd 
import pandas as pd
import folium
from folium import plugins
import leafmap.foliumap as leafmap

DATA_DIR         = Path("../data")

FIRE_CSV_PATH    = DATA_DIR / "processed" / "fire_2023_clean.csv"
FIRE_GEOJSON_PATH   = DATA_DIR / "processed" / "fire_2023_points.geojson"

CITIES_CSV_PATH  = DATA_DIR / "processed" / "selected_cities_canada.csv"
CITIES_GEOJSON_PATH = DATA_DIR / "processed" / "selected_cities_canada.geojson"

LIVE_FIRE_PATH   = DATA_DIR / "processed" / "fire_live.csv"

MAP_OUTPUT_PATH  = Path("../outputs") / "interactive_firemap.html"

### a) Analysis parameters

In [2]:
# EPSG:4326 in geographic degrees; required by Folium and for GeoJSON export.
# EPSG:3347 in metres; required for buffer & spatial join.

CRS_DEFAULT_DEGREE = "EPSG: 4326"
CRS_COUNTRY_METRIC = "EPSG: 3347"

#Storing the symbology characteristics to change it just once for both heatmaps
#HM means HeatMap.

HM_RADIUS = 13 
HM_BLUR = 8
HM_MIN_OPACITY = 0.35

#inferno-inspired colour ramp, where dark > light means low > high intensity.
HM_GRADIENT = {
    0.00: "#000004",
    0.20: "#420A68",
    0.40: "#6A176E",
    0.60: "#932667",
    0.80: "#DD513A",
    1.00: "#FCFFA4",
}

In [3]:
#Later cities accumulate a fire score from number of fires within this radius
BUFFER_KM = 200

#Setup of fire_score thresholds which define for each city the impact level.
#S therefore means Score
S_LOW = 20
S_MODERATE = 50
S_HIGH = 256

In [4]:
#Essential variables for NASA FIRMS live fire API data from VIIRS NOAA-20 NRT data
FIRMS_MAPKEY = "b73850f35cfea7f219a3969b6df1d9a6"

CANADA_BBOX = {"north": 73.2267, "south": 44.6401, "west": -140.625, "east": -49.2188}

In [5]:
#For layering the three toggleable layers we need to set up first a basemap for Canada.
MAP_CENTER = [58.03,-90.82]
ZOOM_START = 4
TILE_STYLE = "CartoDB.DarkMatterNoLabels"

### b) Load Processed Data

In [6]:
fire_2023      = pd.read_csv(FIRE_CSV_PATH)
cities_df      = pd.read_csv(CITIES_CSV_PATH)

assert len(fire_2023) > 0,  "Failed; you may re-run the data cleaning notebook."
assert len(cities_df) > 0,  "Failed; you may re-run the city selection notebook."

print(f"Loaded {len(fire_2023)} wildfire events and {len(cities_df)} cities of Canada")

Loaded 964 wildfire events and 40 cities of Canada


## 1. Computation of GeoDataFrames & CRS Reprojection

In [7]:
#For visualization in Folium we need the data in a GeoDataFrame and as a GeoJSON file
fire_2023_gdf = gpd.GeoDataFrame(
    fire_2023, geometry=gpd.points_from_xy(
        fire_2023.longitude, fire_2023.latitude),
    crs= CRS_DEFAULT_DEGREE)

fire_2023_gdf.to_file(FIRE_GEOJSON_PATH, driver="GeoJSON")

selected_cities_gdf = gpd.GeoDataFrame(
    cities_df,
    geometry=gpd.points_from_xy(
        cities_df.lng,
        cities_df.lat),
    crs= CRS_DEFAULT_DEGREE)

selected_cities_gdf.to_file(CITIES_GEOJSON_PATH, driver="GeoJSON")

#For operations based on distance metric projection is required
cities_metric_gdf    = selected_cities_gdf.to_crs(CRS_COUNTRY_METRIC)
fire_2023_metric_gdf = fire_2023_gdf.to_crs(CRS_COUNTRY_METRIC)

## 2. Spatial Join & Fire Scores for Cities

### a) Setting up buffers around cities

In [8]:
#Buffer the city to then count the fires inside it for impact score
cities_buffered = cities_metric_gdf.copy()
cities_buffered["geometry"] = cities_buffered.geometry.buffer(
    BUFFER_KM * 1000)

### b) Spatial Join

In [9]:
#A Spatial Join is needed to find all fires inside the buffer 
#Since we want to have also cities with no fires within their buffer
#we make a left join.

joined_data = gpd.sjoin(
    cities_buffered, 
    fire_2023_metric_gdf,
    how="left", #cities without fires should be visible
    predicate = "contains"
)

### c) Aggregation fire score per city

In [10]:
#Aggregating fire scores per city to gain an impact score. 
#The class_weight was set up to consider the severity.

fire_per_city = (
    joined_data
    .groupby("city")["class_weight"]
    .sum()
    .reset_index()
    .rename(columns={"class_weight": "fire_score"})
)

#Merging the scores back gives us all relevant data together and by
#converting it into the geographic CRS it can be later visualized. 

selected_cities_gdf = (
    cities_metric_gdf
        .merge(fire_per_city, on="city", how="left")
        .to_crs(CRS_DEFAULT_DEGREE)
)

#For replacing the NaNs (Cities with no fires in their buffer)
#we fill it up with a score of 0 

selected_cities_gdf["fire_score"] = selected_cities_gdf[
    "fire_score"].fillna(0)

## 3. Classification of cities impact levels

In [11]:
def impact(fire_score):
    """
    Classifies the cumulative fire score of a city into wildfire impact

    The thresholds are defined by the constants SCORE_LOW,
    SCORE_MODERATE, and SCORE_HIGH. Those are the sums of fire class_weight
    values (1 / 2 / 4 / 16 / 256) from the NFDB cleaning step.

    Parameters
    ----------
    fire_score_1
        Cumulative class_weight of all fires within CITY_BUFFER_M of a city. 

    Returns
    -------
        One of: "No impact", "Low impact", "Moderate impact", "High impact", "Very high impact".
    
    """
    if fire_score == 0:
        return "No impact"
    if fire_score  < S_LOW:
        return "Low impact"
    if fire_score < S_MODERATE:
        return "Moderate impact"
    if fire_score < S_HIGH:
        return "High impact"
    return "Very high impact"


selected_cities_gdf["impact"] = selected_cities_gdf[
    "fire_score"].apply(impact)

print(selected_cities_gdf["impact"].value_counts())

impact
No impact           13
Low impact          11
Very high impact     6
Moderate impact      6
High impact          4
Name: count, dtype: int64


## 4. Live fire data from NASA FIRMS

In [12]:
today = date.today()
api_url = f"https://firms.modaps.eosdis.nasa.gov/usfs/api/area/csv/{FIRMS_MAPKEY}/VIIRS_NOAA20_NRT/world/5/{today}"

response = requests.get(api_url)

if response.status_code == 200:
    fire_live_df = pd.read_csv(api_url)
    fire_live_df.to_csv(LIVE_FIRE_PATH, index=False)

    fire_live_df = fire_live_df[
        (fire_live_df["latitude"] >= CANADA_BBOX["south"]) &
        (fire_live_df["latitude"] <= CANADA_BBOX["north"]) &
        (fire_live_df["longitude"] >= CANADA_BBOX["west"]) &
        (fire_live_df["longitude"] <= CANADA_BBOX["east"])
        ]
    print(f"Live fire records for Canada: {len(fire_live_df)}")
else:
    print("Error")


Live fire records for Canada: 32


## 5. Put together the map layers

### a) Base map

In [13]:
#By using leafmap we make sure that we can add a legend.
canada_map = leafmap.Map(
    location = MAP_CENTER,
    zoom_start = ZOOM_START,
    tiles = TILE_STYLE, 
)

### b) Weighted Heat Map for wildfire events in 2023

In [14]:
#Heatmap for weighted fires
heat_data = [[point.xy[1][0], point.xy[0][0], weight] 
             for point, weight in zip (
                 fire_2023_gdf.geometry,
                 fire_2023_gdf["class_weight"])
                 ]

plugins.HeatMap(
    heat_data,
    radius=HM_RADIUS,
    blur=HM_BLUR,
    min_opacity=HM_MIN_OPACITY,
    name = "Wildfires in 2023",
    gradient=  HM_GRADIENT
).add_to(canada_map)

### c) Visualization of impact on selected cities

In [15]:
impacted_cities_layer = folium.FeatureGroup(name = "Impacted Cities in 2023")

for city,row in selected_cities_gdf.iterrows(): 
    fire_score = row["fire_score"]
    if fire_score == 0:
        city_color = "#E0FFFF"
        city_radius = 4
        
    elif fire_score > 0 and fire_score < 20:
        city_color = "#87CEFA"
        city_radius = 5
        
    elif fire_score >= 20 and fire_score < 50:
        city_color = "#1E90FF"
        city_radius = 7
        
    elif fire_score >= 50 and fire_score < 256:
        city_color = "#0000FF"
        city_radius = 8
        
    else:
        city_color = "#191970"
        city_radius = 10

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius = city_radius,
        color = "gray",
        weight = 0.9,
        fill = True,
        fill_color=city_color,
        fill_opacity = 0.95,
        tooltip= (f"Name: {row['city']} <br>"
                  f"Country: {row['country']}<br>"
                  f"Population: {row['population']} <br>"
                  f"Impact: {row['impact']}"),
    ).add_to(impacted_cities_layer),
    
impacted_cities_layer.add_to(canada_map)

### d) HeatMap for NASA FIRMS recent data

In [16]:
live_heatmap = [
    [row["latitude"], row["longitude"]]
    for _, row in fire_live_df.iterrows()
] 

plugins.HeatMap(
    live_heatmap,
    radius=HM_RADIUS,
    blur=HM_BLUR,
    min_opacity=HM_MIN_OPACITY,
    name = "Live wildfire",
    show = False,
    gradient= HM_GRADIENT
).add_to(canada_map)

### e) Legend and Layer control

In [17]:
legend_impact ={
    "No Impact": "#E0FFFF",
    "Low Impact": "#87CEFA",
    "Medium Impact": "#1E90FF",
    "High Impact": "#0000FF",
    "Very High Impact": "#191970"
}

canada_map.add_legend(title="Fire impact on city", 
                      legend_dict=legend_impact)

folium.LayerControl(collapsed=False).add_to(canada_map)

## 6. Export of the final map

In [18]:
canada_map.save(MAP_OUTPUT_PATH)

print(f"Map saved to:  {MAP_OUTPUT_PATH}")

Map saved to:  ..\outputs\interactive_firemap.html


## Display Map

In [23]:
#Visualization of Impact Map for Wildfire season 2023
canada_map

In [22]:
#Visualization of recent fires (last actualization 22-05-2026 at 11.00am
canada_map